In [ ]:
import os

import numpy as np
import cv2
import matplotlib.pyplot as plt

dir_path = "./data"

### Masking


In [ ]:
cat = cv2.imread(os.path.join(dir_path, "green_screen_cat.png"))
cat = cv2.cvtColor(cat, cv2.COLOR_BGR2RGB)

plt.imshow(cat)

In [ ]:
green_pixel = cat[0, 0].astype(int)  # top-left pixel

# a pixel is defined by its RGB values, which are three integers in the range [0, 255]
print(green_pixel)

#### Extract the cat by masking out all green pixels


In [ ]:
height, width, channels = cat.shape
# remember that the shape of an image is (height, width, channels)

mask = np.zeros((height, width), dtype=bool)  # create a boolean mask of the same size

mask[:] = False  # initialize the mask to False

for y in range(height):
    for x in range(width):
        pixel = cat[y, x]

        if all(pixel == green_pixel):
            # set the mask to True for pixels that match the green pixel exactly
            mask[y, x] = True

plt.imshow(mask)

#### Um...what is this? This is unexpected.

But some corner areas are masked out.

Maybe I should try relaxing the condition (e.g. not exact matching but with some tolerance)


In [ ]:
mask[:] = 0

for y in range(height):
    for x in range(width):
        pixel = cat[y, x]
        diff = np.abs(pixel - green_pixel)

        if all(diff < 10):  # adjust the threshold yourself
            mask[y, x] = True

plt.imshow(mask)

#### One-liner


In [ ]:
green_mask = np.all(np.abs(cat - green_pixel) < 35, axis=-1)
# The errors come from image noise, compression artifacts, or slight variations in lighting and color.

plt.imshow(green_mask)

#### Let's put the cat on the mountain


In [ ]:
harbour = cv2.imread(os.path.join(dir_path, "hk_harbour.jpg"))
harbour = cv2.cvtColor(harbour, cv2.COLOR_BGR2RGB)

y_start = 200
y_end = y_start + height

x_start = 100
x_end = x_start + width

crop_harbour = harbour[y_start:y_end, x_start:x_end]

# The shapes of crop_harbour and cat should be the same for the masking to work correctly.
print(f"Shape of crop_harbour: {crop_harbour.shape}")
print(f"Shape of cat: {cat.shape}")

plt.imshow(crop_harbour)

In [ ]:
harbour_copy = harbour.copy()
# we will modify harbour_copy instead of harbour to keep the original image intact

crop_harbour = harbour_copy[y_start:y_end, x_start:x_end]

crop_harbour[green_mask] = cat[green_mask]

plt.imshow(harbour_copy)

#### Oops...it obviously is reversed.


In [ ]:
harbour_copy = harbour.copy()

crop_harbour = harbour_copy[y_start:y_end, x_start:x_end]

# "~" is the bitwise NOT operator, which inverts the boolean values in the mask
crop_harbour[~green_mask] = cat[~green_mask]

plt.imshow(harbour_copy)

#### The cat is glowing green. Let's fix it with a stricter threshold.


In [ ]:
stricter_green_mask = np.all(np.abs(cat - green_pixel) < 120, axis=-1)

harbour_copy = harbour.copy()

crop_harbour = harbour_copy[y_start:y_end, x_start:x_end]

crop_harbour[~stricter_green_mask] = cat[~stricter_green_mask]

plt.imshow(harbour_copy)  # Perfect